# Proyecto 2: ChurnInsight — Predicción de Cancelación de Clientes


## Descripción del proyecto

* El desafío de ChurnInsight es crear una solución que prediga si un cliente es propenso a cancelar un servicio (churn).

* El objetivo es que el equipo de Data Science desarrolle un modelo predictivo y que el equipo de Back-end construya una API para disponibilizar esa predicción a otros sistemas, permitiendo que el negocio actúe antes de que el cliente decida irse.

* Ejemplo: una fintech quiere saber, basándose en los hábitos de uso e historial de pago, qué clientes tienen alta probabilidad de deserción. Con esta información, el equipo de marketing puede ofrecer servicios personalizados y el equipo de soporte puede actuar preventivamente.







# **EDA mínimo y limpieza de datos**
## **Proyecto: Churn de clientes**


###VISIÓN GENERAL

*Importar librerias

*Cargar datos

*Inspección mínima

*Identificar variable objetivo

*Detectar valores nulos

*Detectar valores fuera de rango

*Limpiar datos

*Validar dataset final

*Exportar para backend

##Importar librerias

In [2]:
import pandas as pd
import numpy as np

##Cargar datos

In [ ]:
df = pd.read_csv("/credit_card_churn.csv")

##Inspección mínima
####-Detectar números que están como object y columnas que deberían ser numéricas pero no lo son.  

In [ ]:
df.head() #Estructura general
df.shape #Dimensión del dataset
df.info() #Tipos de datos

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10127 entries, 0 to 10126
Data columns (total 23 columns):
 #   Column                                                                                                                              Non-Null Count  Dtype  
---  ------                                                                                                                              --------------  -----  
 0   CLIENTNUM                                                                                                                           10127 non-null  int64  
 1   Attrition_Flag                                                                                                                      10127 non-null  object 
 2   Customer_Age                                                                                                                        10127 non-null  int64  
 3   Gender                                                                           

##Variable objetivo
####-Revisar si hay desbalance, y que columnas no borrar.

In [ ]:
df["Attrition_Flag"].value_counts()

,count
Attrition_Flag,
Existing Customer,8500
Attrited Customer,1627


##Valores nulos

In [ ]:
df.isnull().sum()

,0
CLIENTNUM,0
Attrition_Flag,0
Customer_Age,0
Gender,0
Dependent_count,0
Education_Level,0
Marital_Status,0
Income_Category,0
Card_Category,0
Months_on_book,0


##Estadisticas básicas - Detectar valores fuera de rango

In [ ]:
df.describe()

,CLIENTNUM,Customer_Age,Dependent_count,Months_on_book,Total_Relationship_Count,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2
count,1.012700e+04,10127.000000,10127.000000,10127.000000,10127.000000,10127.000000,10127.000000,10127.000000,10127.000000,10127.000000,10127.000000,10127.000000,10127.000000,10127.000000,10127.000000,10127.000000,10127.000000
mean,7.391776e+08,46.325960,2.346203,35.928409,3.812580,2.341167,2.455317,8631.953698,1162.814061,7469.139637,0.759941,4404.086304,64.858695,0.712222,0.274894,0.159997,0.840003
std,3.690378e+07,8.016814,1.298908,7.986416,1.554408,1.010622,1.106225,9088.776650,814.987335,9090.685324,0.219207,3397.129254,23.472570,0.238086,0.275691,0.365301,0.365301
min,7.080821e+08,26.000000,0.000000,13.000000,1.000000,0.000000,0.000000,1438.300000,0.000000,3.000000,0.000000,510.000000,10.000000,0.000000,0.000000,0.000008,0.000420
25%,7.130368e+08,41.000000,1.000000,31.000000,3.000000,2.000000,2.000000,2555.000000,359.000000,1324.500000,0.631000,2155.500000,45.000000,0.582000,0.023000,0.000099,0.999660
50%,7.179264e+08,46.000000,2.000000,36.000000,4.000000,2.000000,2.000000,4549.000000,1276.000000,3474.000000,0.736000,3899.000000,67.000000,0.702000,0.176000,0.000181,0.999820
75%,7.731435e+08,52.000000,3.000000,40.000000,5.000000,3.000000,3.000000,11067.500000,1784.000000,9859.000000,0.859000,4741.000000,81.000000,0.818000,0.503000,0.000337,0.999900
max,8.283431e+08,73.000000,5.000000,56.000000,6.000000,6.000000,6.000000,34516.000000,2517.000000,34516.000000,3.397000,18484.000000,139.000000,3.714000,0.999000,0.999580,0.999990


####Detectar columnas diferentes o sospechosas.

In [ ]:
df.select_dtypes(include="object").columns

Index(['Attrition_Flag', 'Gender', 'Education_Level', 'Marital_Status',
       'Income_Category', 'Card_Category'],
      dtype='object')

####Revision manual

In [ ]:
df["Credit_Limit"].head()

,Credit_Limit
0,12691.0
1,8256.0
2,3418.0
3,3313.0
4,4716.0


####Valores como 4K,5K,3.5K no son numéricos.

##Limpieza de Datos

In [ ]:
def limpiar_numeros(calor):
  if pd.isna(calor):
    return np.nan

  valor = str(calor).upper().replace(",", "").strip()

  if "K" in valor:
    return float(valor.replace("K", "")) * 1_000
  elif "M" in valor:
    return float(valor.replace("M", "")) * 1_000_000
  else:
    try:
       return float(valor)
    except:
          return np.nan

####Aplicar limpieza a columnas numéricas
#####-cambio de 4K a 4000.0

In [ ]:
df["Credit_Limit"] = df["Credit_Limit"].apply(limpiar_numeros)

####Limpieza de Valores nulos

In [ ]:
#Numéricos a mediana
df["Credit_Limit"] = df["Credit_Limit"].fillna(df["Credit_Limit"].median(), inplace=True)

/tmp/ipython-input-1543107435.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Credit_Limit"] = df["Credit_Limit"].fillna(df["Credit_Limit"].median(), inplace=True)


In [ ]:
#Categóricos a valor explícito
df["Education_Level"].fillna("Unknown", inplace=True)

/tmp/ipython-input-683470821.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Education_Level"].fillna("Unknown", inplace=True)


####Eliminar Columnas inservibles

In [ ]:
df.drop(columns=["CLIENTNUM"], inplace=True)

####Validación final

In [ ]:
df.info()
df.head()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10127 entries, 0 to 10126
Data columns (total 22 columns):
 #   Column                                                                                                                              Non-Null Count  Dtype  
---  ------                                                                                                                              --------------  -----  
 0   Attrition_Flag                                                                                                                      10127 non-null  object 
 1   Customer_Age                                                                                                                        10127 non-null  int64  
 2   Gender                                                                                                                              10127 non-null  object 
 3   Dependent_count                                                                  

,0
Attrition_Flag,0
Customer_Age,0
Gender,0
Dependent_count,0
Education_Level,0
Marital_Status,0
Income_Category,0
Card_Category,0
Months_on_book,0
Total_Relationship_Count,0


####Exportar para el Backend

In [ ]:
df.to_csv("dataset_churn_limpio.csv", index=False)

####Limpiar Income_Category

In [ ]:
df["Income_Category"].value_counts()

,count
Income_Category,
Less than $40K,3561
$40K - $60K,1790
$80K - $120K,1535
$60K - $80K,1402
Unknown,1112
$120K +,727


####Función para limpiar Valores con K

In [ ]:
import numpy as np
import pandas as pd
import re

def convertir_k_a_numero(valor):
    if pd.isna(valor) or (isinstance(valor, str) and valor.upper().strip() == "UNKNOWN"):
        return np.nan

    valor_str = str(valor).upper().strip()

    if "LESS THAN $40K" in valor_str:
        return 39999.0 # Un valor representativo de poco menos de 40K
    elif "$120K +" in valor_str:
        return 120000.0

# Para rangos como "$40K - $60K" o valores únicos como "$40K"
# Extraer la primera parte numérica que podría tener "K"
    matches = re.findall(r'\$?(\d+)K?', valor_str)
    if matches:
        first_num_str = matches[0].replace('K', '')
        return float(first_num_str) * 1_000

    # Respaldo para formatos inesperados, aunque no esperados para los datos actuales
    try:
        return float(valor_str)
    except ValueError:
        return np.nan

In [ ]:
##Aplicar la limpieza:
df["Income_Category_Limpio"] = df["Income_Category"].apply(convertir_k_a_numero)

In [ ]:
#Validar resultado:
df[["Income_Category", "Income_Category_Limpio"]].head()

,Income_Category,Income_Category_Limpio
0,$60K - $80K,60000.0
1,Less than $40K,39999.0
2,$80K - $120K,80000.0
3,Less than $40K,39999.0
4,$60K - $80K,60000.0


In [ ]:
##Eliminar columna original
df.drop(columns=["Income_Category"], inplace=True, errors='ignore')

####Revisar Columna Naive_Bayes_Classifier_

In [ ]:
[col for col in df.columns if "Naive_Bayes" in col]

['Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1',
 'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2']

In [ ]:
#Eliminar columna
df.drop(
    columns=[col for col in df.columns if "Naive_Bayes" in col],
    inplace=True
)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10127 entries, 0 to 10126
Data columns (total 20 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Attrition_Flag            10127 non-null  object 
 1   Customer_Age              10127 non-null  int64  
 2   Gender                    10127 non-null  object 
 3   Dependent_count           10127 non-null  int64  
 4   Education_Level           10127 non-null  object 
 5   Marital_Status            10127 non-null  object 
 6   Card_Category             10127 non-null  object 
 7   Months_on_book            10127 non-null  int64  
 8   Total_Relationship_Count  10127 non-null  int64  
 9   Months_Inactive_12_mon    10127 non-null  int64  
 10  Contacts_Count_12_mon     10127 non-null  int64  
 11  Credit_Limit              0 non-null      object 
 12  Total_Revolving_Bal       10127 non-null  int64  
 13  Avg_Open_To_Buy           10127 non-null  float64
 14  Total_

###Exportar para el backend

In [ ]:
df.to_csv("dataset_churn_limpio_backend.csv", index=False)